[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/signal38/signal38.github.io/blob/main/notebooks/02_finetune.ipynb)

# Notebook 02 — LFM2-350M QLoRA Fine-tuning

Fine-tunes LFM2-350M on Claude-distilled (and ACLED-grounded) NK risk assessment examples using QLoRA.

**Requirements:** T4 GPU runtime, `GITHUB_TOKEN_SIGNAL38` Colab secret.

**Outputs (published to `main`):**
- `models/lfm2-nk-risk/lora_adapter/` — LoRA adapter weights

**Note:** The first run installs dependencies and restarts the Colab runtime. After restart, run all cells again — installation is skipped (sentinel file present).

In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO = Path('/content/signal38.github.io')
if not REPO.exists():
    env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
    subprocess.run(['git', 'clone', '--depth=1', 'https://github.com/signal38/signal38.github.io.git', str(REPO)], check=True, env=env)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.colab_utils import ensure_notebook_requirements
ensure_notebook_requirements('02_finetune', requirements_path=REPO / 'requirements.txt')
# If this is the first run, the runtime restarts here.
# After restart, run all cells again — the sentinel skips reinstall.

In [ ]:
import subprocess, sys, os, json, re
from pathlib import Path

REPO = Path('/content/signal38.github.io')
if not REPO.exists():
    env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
    subprocess.run(['git', 'clone', '--depth=1', 'https://github.com/signal38/signal38.github.io.git', str(REPO)], check=True, env=env)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.colab_utils import prepare_notebook
REPO, PATHS = prepare_notebook(REPO, pull_latest=True)
print(f'Repo ready at {REPO}')

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048
LORA_R = 16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/lfm2-350m',
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # auto-detect: fp16 on T4
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_R,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
model.print_trainable_parameters()

In [ ]:
from datasets import Dataset

def load_and_format(split_path):
    examples = []
    for line in Path(split_path).read_text().splitlines():
        if not line.strip():
            continue
        ex = json.loads(line)
        text = tokenizer.apply_chat_template(
            ex['messages'],
            tokenize=False,
            add_generation_prompt=False,
        )
        examples.append({'text': text})
    return Dataset.from_list(examples)

train_dataset = load_and_format(PATHS['training_dir'] / 'train.jsonl')
val_dataset   = load_and_format(PATHS['training_dir'] / 'val.jsonl')

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}')
print('Sample (first 200 chars):')
print(train_dataset[0]['text'][:200])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

OUTPUT_DIR = str(PATHS['adapter_dir'].parent / 'checkpoints')

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        optim='adamw_8bit',
        weight_decay=0.01,
        lr_scheduler_type='cosine',
        seed=42,
        output_dir=OUTPUT_DIR,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        greater_is_better=False,
        report_to='none',
    ),
)

trainer_stats = trainer.train()
print(f'Training complete. Runtime: {trainer_stats.metrics["train_runtime"]:.1f}s')
print(f'Peak memory: {trainer_stats.metrics.get("train_runtime", "n/a")}')

In [ ]:
adapter_dir = PATHS['adapter_dir']
adapter_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f'Adapter saved to {adapter_dir}')
print('Files:', [f.name for f in adapter_dir.iterdir()])

In [ ]:
# Quick sanity check — one inference pass on the first val example
FastLanguageModel.for_inference(model)

val_example = json.loads(Path(PATHS['training_dir'] / 'val.jsonl').read_text().splitlines()[0])
prompt_messages = [m for m in val_example['messages'] if m['role'] in ('system', 'user')]

prompt = tokenizer.apply_chat_template(
    prompt_messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    temperature=0.1,
    do_sample=True,
)

generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print('Raw model output:')
print(generated)

In [ ]:
from scripts.colab_utils import publish_artifacts

# Collect adapter files
adapter_files = [
    f'models/lfm2-nk-risk/lora_adapter/{f.name}'
    for f in PATHS['adapter_dir'].iterdir()
    if f.is_file()
]

publish_artifacts(
    paths=adapter_files,
    message='Add LFM2-350M QLoRA adapter for NK risk assessment',
    repo_dir=REPO,
)